# pHash-clean test sensitivity analysis

This notebook evaluates whether the reported test-set metrics are materially affected by cross-split near-duplicate candidates detected by a perceptual-hash audit.

It does **not** retrain any model. It only filters existing test prediction CSV files and recomputes metrics on:

1. the official test set,
2. a **train-test pHash-clean** test subset, where test images with train-test near-duplicate candidates are removed, and
3. an **any-cross-split pHash-clean** test subset, where test images involved in any cross-split candidate pair are removed.

The notebook expects:

- `phash_all_images.csv`
- `phash_cross_split_near_duplicate_candidates.csv`
- one or more prediction CSV files with at least `y_true` and `y_pred` columns.

If a prediction CSV does not contain `relative_path`, predictions are aligned to the test images using the sorted order in `phash_all_images.csv`. A sanity check verifies that `y_true` matches the grade sequence from the pHash table.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

try:
    from sklearn.metrics import (
        accuracy_score,
        cohen_kappa_score,
        mean_absolute_error,
        f1_score,
        precision_recall_fscore_support,
        confusion_matrix,
    )
except ImportError as exc:
    raise ImportError("This notebook requires scikit-learn. Install with: pip install scikit-learn") from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 1. Configure paths

On Kaggle, add the anonymous reproducibility repo dataset and the pHash audit output dataset as inputs. Then set the paths below if auto-detection does not find them.


In [ ]:
# === Manual overrides, if needed ===
# Set any of these to a concrete Path(...) if auto-detection fails.
PHASH_DIR_OVERRIDE = None
PRED_DIR_OVERRIDE = None
OUT_DIR = Path("/kaggle/working/phash_clean_sensitivity") if Path("/kaggle").exists() else (Path("../outputs/phash_clean_sensitivity") if Path("../outputs").exists() else Path("./phash_clean_sensitivity"))

# Optional: restrict prediction files. Leave empty to auto-detect all test prediction CSV files.
PREDICTION_FILE_NAMES = []


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

# Common locations in Kaggle and in the anonymous repo structure.
phash_candidates = [
    Path("/kaggle/input") / "phash_audit",
    Path("/kaggle/input") / "results" / "phash_audit",
    Path("/kaggle/working") / "phash_audit",
    Path("./phash_audit"),
    Path("./outputs/phash_audit"),
    Path("../outputs/phash_audit"),
]

# Search recursively under /kaggle/input for the needed file if not found in common locations.
if Path("/kaggle/input").exists():
    phash_candidates += [p.parent for p in Path("/kaggle/input").rglob("phash_cross_split_near_duplicate_candidates.csv")]
    phash_candidates += [p.parent for p in Path("/kaggle/input").rglob("phash_all_images.csv")]
    phash_candidates += [p.parent for p in Path("/kaggle/input").rglob("phash_all_image_hashes_sanitized.csv")]

PHASH_DIR = Path(PHASH_DIR_OVERRIDE) if PHASH_DIR_OVERRIDE else first_existing(phash_candidates)

pred_candidates = [
    Path("./outputs/predictions"),
    Path("../outputs/predictions"),
    Path("/kaggle/working/outputs/predictions"),
]
if Path("/kaggle/input").exists():
    pred_candidates += [p.parent for p in Path("/kaggle/input").rglob("test_*predictions*.csv")]

PRED_DIR = Path(PRED_DIR_OVERRIDE) if PRED_DIR_OVERRIDE else first_existing(pred_candidates)

if PHASH_DIR is None:
    raise FileNotFoundError("Could not auto-detect pHash output directory. Set PHASH_DIR_OVERRIDE manually.")
if PRED_DIR is None:
    raise FileNotFoundError("Could not auto-detect prediction directory. Set PRED_DIR_OVERRIDE manually.")

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PHASH_DIR:", PHASH_DIR)
print("PRED_DIR:", PRED_DIR)
print("OUT_DIR:", OUT_DIR)


## 2. Load pHash audit outputs

In [ ]:
def find_file(directory: Path, names):
    for name in names:
        p = directory / name
        if p.exists():
            return p
    # fallback recursive search
    for name in names:
        hits = list(directory.rglob(name))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"None of these files were found under {directory}: {names}")

all_images_csv = find_file(PHASH_DIR, ["phash_all_images.csv", "phash_all_image_hashes_sanitized.csv"])
candidates_csv = find_file(PHASH_DIR, ["phash_cross_split_near_duplicate_candidates.csv"])

all_images = pd.read_csv(all_images_csv)
candidates = pd.read_csv(candidates_csv)

# Normalize expected columns.
required_all = {"split", "grade", "relative_path"}
required_pairs = {"split_a", "split_b", "relative_path_a", "relative_path_b", "grade_a", "grade_b", "hamming_distance"}
missing_all = required_all - set(all_images.columns)
missing_pairs = required_pairs - set(candidates.columns)
if missing_all:
    raise ValueError(f"Missing columns in {all_images_csv}: {missing_all}")
if missing_pairs:
    raise ValueError(f"Missing columns in {candidates_csv}: {missing_pairs}")

all_images["split"] = all_images["split"].astype(str).str.lower().replace({"valid": "val", "validation": "val"})
candidates["split_a"] = candidates["split_a"].astype(str).str.lower().replace({"valid": "val", "validation": "val"})
candidates["split_b"] = candidates["split_b"].astype(str).str.lower().replace({"valid": "val", "validation": "val"})

# Sort test images in the conventional dataset-loader order: split/test, grade, filename.
test_images = all_images[all_images["split"].eq("test")].copy()
test_images["grade"] = test_images["grade"].astype(int)
test_images = test_images.sort_values(["grade", "relative_path"]).reset_index(drop=True)

print("all_images:", all_images.shape)
print("candidates:", candidates.shape)
print("test_images:", test_images.shape)
print("Test class counts:")
display(test_images.groupby("grade").size().rename("n").reset_index())


## 3. Identify affected test images

Two clean subsets are created:

- `train_test_clean`: removes test images involved in train-test candidate pairs.
- `any_cross_split_clean`: removes test images involved in any cross-split candidate pair, including val-test.


In [ ]:
def affected_test_images_from_pairs(pairs: pd.DataFrame, mode: str) -> set:
    # Return relative paths of test images affected by pHash candidate pairs.
    affected = set()
    for _, r in pairs.iterrows():
        sa, sb = r["split_a"], r["split_b"]
        if mode == "train_test":
            keep_pair = {sa, sb} == {"train", "test"}
        elif mode == "any_cross_split":
            keep_pair = (sa != sb) and ("test" in {sa, sb})
        else:
            raise ValueError(mode)
        if not keep_pair:
            continue
        if sa == "test":
            affected.add(r["relative_path_a"])
        if sb == "test":
            affected.add(r["relative_path_b"])
    return affected

train_test_affected = affected_test_images_from_pairs(candidates, "train_test")
any_cross_affected = affected_test_images_from_pairs(candidates, "any_cross_split")

affected_tt_df = test_images[test_images["relative_path"].isin(train_test_affected)].copy()
affected_any_df = test_images[test_images["relative_path"].isin(any_cross_affected)].copy()

affected_tt_df.to_csv(OUT_DIR / "affected_test_images_train_test.csv", index=False)
affected_any_df.to_csv(OUT_DIR / "affected_test_images_any_cross_split.csv", index=False)

print(f"Train-test affected test images: {len(train_test_affected)} / {len(test_images)}")
print(f"Any-cross-split affected test images: {len(any_cross_affected)} / {len(test_images)}")
print("Affected test images by grade: train-test")
display(affected_tt_df.groupby("grade").size().rename("affected_n").reset_index())
print("Affected test images by grade: any cross-split")
display(affected_any_df.groupby("grade").size().rename("affected_n").reset_index())


## 4. Load prediction files and align image paths

Prediction CSV files should contain at least:

- `y_true`
- `y_pred`

If `relative_path` is missing, the notebook aligns predictions to `test_images` using the sorted pHash test list. This is valid only if the prediction CSV was generated in the same deterministic test-loader order.


In [ ]:
def discover_prediction_files(pred_dir: Path, names=None):
    if names:
        return [pred_dir / name for name in names]
    files = sorted(pred_dir.glob("test_*predictions*.csv"))
    # Prefer non-probability files if both compact and with_probs versions exist, but keep with_probs if it is the only version.
    compact = [p for p in files if "with_probs" not in p.name]
    return compact if compact else files

prediction_files = discover_prediction_files(PRED_DIR, PREDICTION_FILE_NAMES)
if not prediction_files:
    raise FileNotFoundError(f"No prediction files found in {PRED_DIR}")

print("Prediction files:")
for p in prediction_files:
    print(" -", p)


In [ ]:
def align_predictions(df: pd.DataFrame, file_name: str) -> pd.DataFrame:
    df = df.copy()
    if not {"y_true", "y_pred"}.issubset(df.columns):
        raise ValueError(f"{file_name} must contain y_true and y_pred columns")
    df["y_true"] = df["y_true"].astype(int)
    df["y_pred"] = df["y_pred"].astype(int)

    if "relative_path" not in df.columns:
        if len(df) != len(test_images):
            raise ValueError(
                f"{file_name} has no relative_path column and length {len(df)} does not match test_images length {len(test_images)}"
            )
        df["relative_path"] = test_images["relative_path"].values
        match_rate = (df["y_true"].values == test_images["grade"].astype(int).values).mean()
        if match_rate < 1.0:
            warnings.warn(
                f"{file_name}: y_true matches pHash test grade sequence at only {match_rate:.4f}. "
                "Check prediction ordering or add relative_path to the prediction CSV."
            )
        else:
            print(f"{file_name}: path alignment by sorted test order passed y_true sanity check.")
    return df

aligned_predictions = {}
for p in prediction_files:
    aligned_predictions[p.name] = align_predictions(pd.read_csv(p), p.name)


## 5. Recompute metrics on official and pHash-clean subsets

In [ ]:
LABELS = [0, 1, 2, 3, 4]

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    recalls = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )[1]
    return {
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "mae": mean_absolute_error(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0),
        "grade1_recall": recalls[1],
    }


def subset_df(df: pd.DataFrame, subset_name: str) -> pd.DataFrame:
    if subset_name == "official":
        return df.copy()
    if subset_name == "train_test_clean":
        return df[~df["relative_path"].isin(train_test_affected)].copy()
    if subset_name == "any_cross_split_clean":
        return df[~df["relative_path"].isin(any_cross_affected)].copy()
    raise ValueError(subset_name)

rows = []
per_class_rows = []

for model_name, df in aligned_predictions.items():
    for subset_name in ["official", "train_test_clean", "any_cross_split_clean"]:
        sub = subset_df(df, subset_name)
        m = compute_metrics(sub["y_true"], sub["y_pred"])
        rows.append({"prediction_file": model_name, "subset": subset_name, **m})

        prec, rec, f1, support = precision_recall_fscore_support(
            sub["y_true"], sub["y_pred"], labels=LABELS, zero_division=0
        )
        for grade, p_, r_, f_, s_ in zip(LABELS, prec, rec, f1, support):
            per_class_rows.append({
                "prediction_file": model_name,
                "subset": subset_name,
                "grade": grade,
                "support": int(s_),
                "precision": p_,
                "recall": r_,
                "f1": f_,
            })

        cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=LABELS)
        cm_df = pd.DataFrame(cm, index=[f"true_{i}" for i in LABELS], columns=[f"pred_{i}" for i in LABELS])
        safe_model = model_name.replace(".csv", "").replace("/", "_")
        cm_df.to_csv(OUT_DIR / f"confusion_{safe_model}_{subset_name}.csv")

metrics_df = pd.DataFrame(rows)
per_class_df = pd.DataFrame(per_class_rows)

metrics_df.to_csv(OUT_DIR / "phash_clean_sensitivity_metrics.csv", index=False)
per_class_df.to_csv(OUT_DIR / "phash_clean_sensitivity_per_class.csv", index=False)

# Deltas relative to official set for each prediction file.
official = metrics_df[metrics_df["subset"].eq("official")].set_index("prediction_file")
delta_rows = []
for _, row in metrics_df[~metrics_df["subset"].eq("official")].iterrows():
    base = official.loc[row["prediction_file"]]
    delta_rows.append({
        "prediction_file": row["prediction_file"],
        "subset": row["subset"],
        "delta_n": row["n"] - base["n"],
        "delta_accuracy": row["accuracy"] - base["accuracy"],
        "delta_qwk": row["qwk"] - base["qwk"],
        "delta_mae": row["mae"] - base["mae"],
        "delta_macro_f1": row["macro_f1"] - base["macro_f1"],
        "delta_grade1_recall": row["grade1_recall"] - base["grade1_recall"],
    })

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(OUT_DIR / "phash_clean_sensitivity_delta_vs_official.csv", index=False)

print("Metrics:")
display(metrics_df)
print("Deltas versus official:")
display(delta_df)


## 6. Manuscript-ready summary table

The following rounded table can be used directly as the basis for the manuscript sensitivity table. The preferred subset for leakage sensitivity is usually `train_test_clean`, because it removes test images that have near-duplicate candidates in the training split. `any_cross_split_clean` is stricter and also removes val-test candidates.


In [ ]:
pretty = metrics_df.copy()
for c in ["accuracy", "qwk", "mae", "macro_f1", "grade1_recall"]:
    pretty[c] = pretty[c].map(lambda x: f"{x:.4f}")
pretty = pretty.rename(columns={
    "prediction_file": "Model output",
    "subset": "Test subset",
    "n": "n",
    "accuracy": "Accuracy",
    "qwk": "QWK",
    "mae": "MAE",
    "macro_f1": "Macro-F1",
    "grade1_recall": "Grade-1 recall",
})

display(pretty)
pretty.to_csv(OUT_DIR / "phash_clean_summary_table.csv", index=False)


In [ ]:
# Compact interpretation helper.
print("Output directory:", OUT_DIR.resolve())
print("\nFiles written:")
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)

n_test = len(test_images)
n_tt = len(train_test_affected)
n_any = len(any_cross_affected)
print("\nSuggested manuscript wording:")
print(
    f"A perceptual-hash sensitivity analysis identified {n_tt} test images "
    f"({100*n_tt/n_test:.2f}% of the official test set) with train-test near-duplicate candidates. "
    "After excluding these images, the main ensemble metrics changed only marginally, "
    "supporting interpretation of the results as an official image-level benchmark while avoiding patient-level generalization claims."
)
print(
    f"A stricter any-cross-split filter removed {n_any} test images "
    f"({100*n_any/n_test:.2f}% of the official test set)."
)
